# Quantization Deep Dive

> **Status:** Content notebook — AWQ and GPTQ exercises require a GPU with sufficient VRAM.

## Learning Objectives

By the end of this notebook, you will be able to:

- [ ] Explain the difference between post-training quantization (PTQ) and quantization-aware training (QAT)
- [ ] Describe the AWQ, GPTQ, EXL2, and GGUF quantization schemes and their trade-offs
- [ ] Load a quantized model using `bitsandbytes`, `AutoAWQ`, or `auto-gptq`
- [ ] Benchmark the quality/performance trade-off of different quantization levels
- [ ] Choose the right quantization scheme for a given hardware and latency target

---

## Prerequisites

- [01_START_HERE.ipynb](../01_START_HERE/01_START_HERE.ipynb)
- [03_kv_cache_paged_attention.ipynb](../03_kv_cache_paged_attention/03_kv_cache_paged_attention.ipynb)
- Basic PyTorch knowledge

---

## 1. Why Quantize?

LLMs store weights as 32-bit (FP32) or 16-bit (BF16/FP16) floats by default.

| Precision | Bytes/param | Llama-3 8B memory | Llama-3 70B memory |
|-----------|-------------|-------------------|--------------------|
| FP32      | 4           | ~32 GB            | ~280 GB            |
| BF16/FP16 | 2           | ~16 GB            | ~140 GB            |
| INT8      | 1           | ~8 GB             | ~70 GB             |
| INT4      | 0.5         | ~4 GB             | ~35 GB             |
| INT2      | 0.25        | ~2 GB             | ~17 GB             |

Quantization lets you run larger models on smaller hardware while trading off some quality.

---

## 2. Quantization Schemes

### bitsandbytes (BnB) — 8-bit and 4-bit
- Simple drop-in quantization via HuggingFace `load_in_8bit=True` / `load_in_4bit=True`
- NF4 (Normal Float 4) data type for 4-bit with double quantization
- Best for: fast experimentation, fine-tuning with LoRA

### GPTQ (Generative Pre-trained Transformer Quantization)
- PTQ using a calibration dataset to minimize layer-wise reconstruction error
- Produces INT4 weights with ~1% quality loss vs FP16
- Compatible with `auto-gptq` and ExLlamaV2
- Best for: inference speed on NVIDIA GPUs

### AWQ (Activation-Aware Weight Quantization)
- Protects the most salient weights (high activation magnitude) from quantization error
- Achieves better quality than GPTQ at the same bit-width
- Supported by `autoawq` and vLLM natively
- Best for: deployment in production vLLM stacks

### EXL2 (ExLlamaV2 format)
- Mixed-precision quantization per-layer using calibration data
- Can specify average bits (e.g., 4.0, 5.0) with per-layer variation
- Highest quality at 4-bit among local inference formats
- Best for: consumer GPU inference with quality priority

### GGUF (llama.cpp format)
- k-quant levels: Q4_K_M, Q5_K_M, Q8_0
- CPU-friendly; runs on Apple Silicon, CPUs, and low-VRAM GPUs
- Best for: Ollama, LM Studio, offline/local use

---

## 3. Loading Quantized Models

```python
# BitsAndBytes 4-bit (requires bitsandbytes, transformers)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
```

```python
# AWQ via AutoAWQ (requires autoawq)
from awq import AutoAWQForCausalLM

model = AutoAWQForCausalLM.from_quantized(
    "casperhansen/llama-3-8b-instruct-awq",
    fuse_layers=True,
)
```

```python
# Serve AWQ model with vLLM
from vllm import LLM, SamplingParams

llm = LLM(
    model="casperhansen/llama-3-8b-instruct-awq",
    quantization="awq",
    dtype="float16",
)
outputs = llm.generate(["Explain quantization in one sentence."], SamplingParams(max_tokens=100))
print(outputs[0].outputs[0].text)
```

---

## 4. Quality Benchmarking

Use `lm-evaluation-harness` to measure perplexity and accuracy across tasks:

```bash
# Install
pip install lm-eval

# Run evaluation on a quantized model
lm_eval --model hf   --model_args pretrained=casperhansen/llama-3-8b-instruct-awq,dtype=float16   --tasks hellaswag,arc_easy,mmlu   --device cuda:0   --batch_size 8
```

Typical quality retention at INT4 (vs BF16): ~97–99% on standard benchmarks.

---

## 5. Choosing the Right Scheme

| Use case | Recommended |
|----------|------------|
| Fine-tuning on consumer GPU | BnB 4-bit (QLoRA) |
| Production vLLM serving | AWQ |
| Maximum quality at 4-bit | GPTQ or EXL2 |
| CPU / Apple Silicon / Ollama | GGUF Q4_K_M or Q5_K_M |
| Minimal code changes | BnB 8-bit |

---

## Exercises

1. Load `meta-llama/Meta-Llama-3-8B-Instruct` in both BF16 and 4-bit NF4 and compare GPU memory usage.
2. Benchmark inference throughput (tokens/sec) for BF16 vs AWQ INT4 on the same prompt batch.
3. Run `lm-evaluation-harness` on a GGUF model and compare MMLU scores vs the full-precision baseline.

---

## References

- [AWQ paper](https://arxiv.org/abs/2306.00978)
- [GPTQ paper](https://arxiv.org/abs/2210.17323)
- [bitsandbytes docs](https://huggingface.co/docs/bitsandbytes)
- [LM Evaluation Harness](https://github.com/EleutherAI/lm-evaluation-harness)

## What Comes Next

- [05_speculative_decoding.ipynb](../05_speculative_decoding/05_speculative_decoding.ipynb) — Speed up decode latency without quality loss
- [06_serving_runtimes_comparison.ipynb](../06_serving_runtimes_comparison/06_serving_runtimes_comparison.ipynb) — Compare vLLM, TRT-LLM, and SGLang
